<a href="https://colab.research.google.com/github/ajabgullb/sentiment-analysis-rnn/blob/main/Sentiment_Analysis_RNN_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -q https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xzf aclImdb_v1.tar.gz

In [2]:
import os
import re
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
  accuracy_score, precision_score, recall_score,
  f1_score, confusion_matrix, classification_report,
  roc_auc_score, roc_curve
)

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import  word_tokenize
from nltk.stem import WordNetLemmatizer

from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

from tensorflow.keras.preprocessing.text import  Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

# Loading the Data

In [4]:
def load_imdb(split_dir):
  texts, labels = [], []

  for label, sentiment in enumerate(["neg", "pos"]):
    folder = os.path.join(split_dir, sentiment)

    for filepath in glob.glob(os.path.join(folder, "*.txt")):
      with open(filepath, "r", encoding="utf-8") as f:
        texts.append(f.read())
        labels.append(label)  # 0 = neg, 1 = pos

  return texts, labels

train_texts, train_labels = load_imdb("aclImdb/train")
test_texts,  test_labels  = load_imdb("aclImdb/test")

# Preprocessing of Text

In [5]:
STOP_WORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
  # 1. Strip HTML tags
  text = re.sub(r"<.*?>", " ", text)
  # 2. Lowercase
  text = text.lower()
  # 3. Remove non-alpha characters (keep spaces)
  text = re.sub(r"[^a-z\s]", " ", text)
  # 4. Collapse multiple spaces
  text = re.sub(r"\s+", " ", text).strip()
  return text


def tokenize_and_clean(text: str) -> list[str]:
  tokens = nltk.word_tokenize(clean_text(text))

  tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in STOP_WORDS]
  tokens = [t for t in tokens if len(t) > 1]   # drop single-char tokens

  return tokens


train_tokens = [tokenize_and_clean(t) for t in train_texts]
test_tokens  = [tokenize_and_clean(t) for t in test_texts]

In [6]:
train_tokens

[['big',
  'fan',
  'old',
  'western',
  'believe',
  'hollywood',
  'capable',
  'capturing',
  'old',
  'glory',
  'even',
  'ronda',
  'fleming',
  'stewart',
  'granger',
  'help',
  'movie',
  'carry',
  'nearly',
  'trite',
  'characteristic',
  'western',
  'reformed',
  'gunfighter',
  'turned',
  'good',
  'guy',
  'fallen',
  'virtuous',
  'woman',
  'bigoted',
  'townspeople',
  'must',
  'turn',
  'gunfighter',
  'salvation',
  'etc',
  'help',
  'notice',
  'last',
  'name',
  'writer',
  'young',
  'actor',
  'play',
  'granger',
  'son',
  'nepotism',
  'seen',
  'better',
  'acting',
  'high',
  'school',
  'play',
  'chill',
  'will',
  'play',
  'cartoon',
  'characterization',
  'chill',
  'will',
  'reached',
  'word',
  'yet'],
 ['watched',
  'movie',
  'yesterday',
  'highly',
  'disappointed',
  'heather',
  'graham',
  'tom',
  'cavanaugh',
  'basically',
  'carry',
  'awkwardly',
  'unbelievable',
  'script',
  'five',
  'hour',
  'however',
  'long',
  'actua

## Training Word2Vec Embeddings - Skip-Gram with Negative Sampling

In [ ]:
!pip install -q gensim
import gensim.downloader as api

w2v_model = api.load("word2vec-google-news-300")

In [8]:
from gensim.models import Word2Vec

EMBEDDING_DIM = 128   # d — dimensionality of each word vector
WINDOW        = 5     # context window size c
MIN_COUNT     = 2     # ignore tokens appearing < 2 times
WORKERS       = 4
EPOCHS        = 10

w2v_model = Word2Vec(
    sentences  = train_tokens,   # only train on training set!
    vector_size= EMBEDDING_DIM,
    window     = WINDOW,
    min_count  = MIN_COUNT,
    workers    = WORKERS,
    sg         = 1,              # 1 = Skip-Gram, 0 = CBOW
    negative   = 5,              # negative sampling k=5
    epochs     = EPOCHS
)

w2v_model.save("word2vec_imdb.model")

In [9]:
# Vocabulary & Index Mapping

# Special tokens
PAD_TOKEN = "<PAD>"  # index 0 — used for padding shorter sequences
UNK_TOKEN = "<UNK>"  # index 1 — used for OOV words at test time

vocab = w2v_model.wv.key_to_index # dict: word → idx

word2idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}
word2idx.update({word: idx + 2 for word, idx in vocab.items()})

VOCAB_SIZE = len(word2idx)
print(f"Vocabulary size: {VOCAB_SIZE}")  # expect ~40K–80K

Vocabulary size: 40978


In [10]:
# Create Embedding Matrix
embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM), dtype=np.float32)

for word, idx in word2idx.items():
  if word in w2v_model.wv:
    embedding_matrix[idx] = w2v_model.wv[word]
  # PAD (idx=0) and UNK (idx=1) remain as zero vectors

In [11]:
embedding_matrix.shape
# expected shape: (40978, 128)

(40978, 128)

In [12]:
# Sequence Encoding

def encode_tokens(token_list: list[str], word2idx: dict) -> list[int]:
  return [word2idx.get(tok, word2idx[UNK_TOKEN]) for tok in token_list]

train_encoded = [encode_tokens(tokens, word2idx) for tokens in train_tokens]
test_encoded  = [encode_tokens(tokens, word2idx) for tokens in test_tokens]

In [ ]:
# Sequence Length Analysis

lengths = [len(s) for s in train_encoded]

print(f"Mean: {np.mean(lengths):.0f}, Median: {np.median(lengths):.0f}, "
      f"95th pct: {np.percentile(lengths, 95):.0f}, Max: {max(lengths)}")

plt.hist(lengths, bins=60)
plt.axvline(np.percentile(lengths, 95), color='r', label='95th percentile')
plt.xlabel("Sequence length"); plt.show()

In [14]:
MAX_SEQ_LEN = int(np.percentile(lengths, 95))

def pad_or_truncate(seq: list[int], max_len: int, pad_idx: int = 0) -> list[int]:
  if len(seq) > max_len:
    return seq[:max_len] # truncate from the end

  return seq + [pad_idx] * (max_len - len(seq))  # post-pad with 0

train_padded = np.array([pad_or_truncate(s, MAX_SEQ_LEN) for s in train_encoded])
test_padded  = np.array([pad_or_truncate(s, MAX_SEQ_LEN) for s in test_encoded])

print(train_padded.shape)  # (25000, 450)

(25000, 309)


In [15]:
# Split the dataset into train/val sets

all_train_lengths = np.array([min(len(s), MAX_SEQ_LEN) for s in train_encoded])

X_train, X_val, y_train, y_val, len_train, len_val = train_test_split(
    train_padded, np.array(train_labels), all_train_lengths,
    test_size=0.1, random_state=42, stratify=train_labels
)

In [16]:
class IMDBDataset(Dataset):
  def __init__(self, X, y):
    self.X = torch.tensor(X, dtype=torch.long)
    self.y = torch.tensor(y, dtype=torch.float32)

  def __len__(self):  return len(self.X)
  def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE = 64

train_dataset = IMDBDataset(X_train, y_train)
val_dataset   = IMDBDataset(X_val,   y_val)
test_dataset  = IMDBDataset(test_padded, np.array(test_labels))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

In [17]:
# Load the Embedding Matrix into PyTorch

embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)

embedding_layer = nn.Embedding(
  num_embeddings = VOCAB_SIZE,
  embedding_dim  = EMBEDDING_DIM,
  padding_idx    = 0           # gradient for PAD token is zeroed out
)

embedding_layer.weight = nn.Parameter(embedding_tensor)
embedding_layer.weight.requires_grad = True  # allow fine-tuning during training

# Training the RNN Architecture on Preprocessed Text

In [18]:
class SentimentRNN (nn.Module):

  def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 embedding_matrix, output_dim=1, num_layers=1,
                 dropout=0.3, pad_idx=0) -> None:
    super().__init__()

    # Embedding layer initialized with pretrained Word2Vec weights
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
    self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float32))
    self.embedding.weight.requires_grad = True # fine-tune during training

    self.rnn = nn.RNN(
      input_size  = embedding_dim,
      hidden_size = hidden_dim,
      num_layers  = num_layers,
      batch_first = True,
      nonlinearity= "tanh",
      dropout     = dropout if num_layers > 1 else 0.0,
      bidirectional = False
    )

    self.dropout = nn.Dropout(dropout)
    self.fc = nn.Linear(hidden_dim, output_dim)


  def forward (self, x, lengths):
    embedded = self.embedding(x)
    packed = nn.utils.rnn.pack_padded_sequence(
      embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
    ) # pack to skip computation over PAD tokens
    packed_out, hidden = self.rnn(packed)
    last_hidden = hidden[-1]

    out = self.dropout(last_hidden)
    logits = self.fc(out)

    return logits.squeeze(1)

class IMDBDataset(Dataset):

  def __init__(self, X, y, true_lengths):
    self.X = torch.tensor(X, dtype=torch.long)
    self.y = torch.tensor(y, dtype=torch.float32)
    self.lengths = torch.tensor(true_lengths, dtype=torch.long)


  def __len__(self): return len(self.X)
  def __getitem__(self, idx):
    return self.X[idx], self.y[idx], self.lengths[idx]


# capture lengths BEFORE padding, capped at MAX_SEQ_LEN
# train_lengths = [min(len(s), MAX_SEQ_LEN) for s in train_encoded]
# val_lengths   = [min(len(s), MAX_SEQ_LEN) for s in [train_encoded[i] for i in train_idx]]
# test_lengths  = [min(len(s), MAX_SEQ_LEN) for s in test_encoded]

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HIDDEN_DIM = 64

model = SentimentRNN(
    vocab_size       = VOCAB_SIZE,
    embedding_dim    = EMBEDDING_DIM,
    hidden_dim       = HIDDEN_DIM,
    embedding_matrix = embedding_matrix,
    num_layers       = 1,
    dropout          = 0.5
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

In [20]:
device

device(type='cuda')

In [21]:
# Training Loop

def train_one_epoch(model, loader, optimizer, criterion, device):
  model.train()
  total_loss, correct, total = 0, 0, 0

  for X_batch, y_batch, lengths in loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    optimizer.zero_grad()

    logits = model(X_batch, lengths)
    loss = criterion(logits, y_batch)
    loss.backward()

    # Gradient clipping — essential for vanilla RNNs to prevent exploding gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    optimizer.step()

    total_loss += loss.item() * X_batch.size(0)
    preds = (torch.sigmoid(logits) >= 0.5).float()
    correct += (preds == y_batch).sum().item()
    total += X_batch.size(0)

  return (total_loss / total), (correct / total)

In [22]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
  model.eval()
  total_loss, correct, total = 0, 0, 0

  for X_batch, y_batch, lengths in loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    logits = model(X_batch, lengths)
    loss = criterion(logits, y_batch)

    total_loss += loss.item() * X_batch.size(0)
    preds = (torch.sigmoid(logits) >= 0.5).float()
    correct += (preds == y_batch).sum().item()
    total += X_batch.size(0)

  return (total_loss / total), (correct / total)

In [ ]:
N_EPOCHS = 10

patience = 3
epochs_without_improvement = 0
best_val_loss = float("inf")

# Re-initialize DataLoaders with correct IMDBDataset arguments
# The IMDBDataset class from MY2En6X0Vx8w is expected to be in scope
# All necessary variables (X_train, y_train, len_train, etc.) are available in kernel state

# Calculate test_lengths, similar to all_train_lengths
test_lengths = np.array([min(len(s), MAX_SEQ_LEN) for s in test_encoded])

train_dataset = IMDBDataset(X_train, y_train, len_train)
val_dataset   = IMDBDataset(X_val, y_val, len_val)
test_dataset  = IMDBDataset(test_padded, np.array(test_labels), test_lengths)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)


for epoch in range(N_EPOCHS):
  start = time.time()
  train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
  val_loss, val_acc = evaluate(model, val_loader, criterion, device)

  if epoch <= 5:
    model.embedding.weight.requires_grad = False
  else:
    model.embedding.weight.requires_grad = True

  if val_loss < best_val_loss:
    best_val_loss = val_loss
    epochs_without_improvement = 0
    torch.save(model.state_dict(), "best_rnn_model.pt")
  else:
    epochs_without_improvement += 1
    if epochs_without_improvement >= patience:
      print(f"Early stopping at epoch {epoch+1}")
      break

  print(f"Epoch {epoch+1}/{N_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
        f"Time: {time.time()-start:.1f}s")

In [ ]:
# Full Evaluation on the Test Set

model.load_state_dict(torch.load("best_rnn_model.pt"))
model.eval()

all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
  for X_batch, y_batch, lengths in test_loader:
    X_batch = X_batch.to(device)
    logits = model(X_batch, lengths)
    probs = torch.sigmoid(logits).cpu()
    preds = (probs >= 0.5).float()

    all_probs.extend(probs.numpy())
    all_preds.extend(preds.numpy())
    all_labels.extend(y_batch.numpy())


print("Test Accuracy :", accuracy_score(all_labels, all_preds))
print("Precision     :", precision_score(all_labels, all_preds))
print("Recall        :", recall_score(all_labels, all_preds))
print("F1 Score      :", f1_score(all_labels, all_preds))
print("ROC-AUC       :", roc_auc_score(all_labels, all_probs))
print("\n", classification_report(all_labels, all_preds, target_names=["neg", "pos"]))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.imshow(cm, cmap="Blues")
plt.xticks([0,1], ["neg","pos"]); plt.yticks([0,1], ["neg","pos"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha="center", va="center")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(all_labels, all_probs):.3f}")
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.legend(); plt.title("ROC Curve"); plt.show()

Visualizing the Vanishing Gradient Problem in RNN

In [ ]:
import torch.nn.utils.rnn
# visualize the vanishing gradient problem

def visualize_gradient_flow(model, X_batch, y_batch, lengths, criterion, device):
    model_was_training = model.training
    model.train()  # Temporarily set to train mode for backward pass with cudnn RNN
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)

    # Store original requires_grad state of embedding weight
    original_embedding_requires_grad = model.embedding.weight.requires_grad
    # Ensure embedding weight requires grad for this visualization
    model.embedding.weight.requires_grad = True

    embedded = model.embedding(X_batch)              # (batch, seq_len, emb_dim)
    embedded.retain_grad()

    # Manually unroll the RNN so we can grab gradients at every timestep
    # Initial hidden state needs requires_grad=True if we want to track its gradients
    h = torch.zeros(1, X_batch.size(0), model.rnn.hidden_size, device=device, requires_grad=True)
    hidden_states = []
    for t in range(embedded.size(1)):
        out, h = model.rnn(embedded[:, t:t+1, :], h)
        h.retain_grad()
        hidden_states.append(h)

    final_hidden = hidden_states[-1][-1]
    logits = model.fc(model.dropout(final_hidden))
    loss = criterion(logits.squeeze(1), y_batch)
    loss.backward()

    # Restore original requires_grad state of embedding weight
    model.embedding.weight.requires_grad = original_embedding_requires_grad

    # Restore model training state
    if not model_was_training:
        model.eval()

    # Gradient norm at each timestep
    grad_norms = [h.grad.norm().item() for h in hidden_states]
    return grad_norms

# Grab a single batch
X_batch, y_batch, lengths = next(iter(train_loader))
grad_norms = visualize_gradient_flow(model, X_batch, y_batch, lengths, criterion, device)

plt.figure(figsize=(10, 4))
plt.plot(grad_norms)
plt.yscale("log")
plt.xlabel("Timestep (forward direction)")
plt.ylabel("Gradient norm at hidden state (log scale)")
plt.title("Actual gradient magnitude across your RNN's hidden states")
plt.gca().invert_xaxis()  # so it reads right-to-left, matching "timesteps back from output"
plt.show()

Inference Pipeline

In [29]:
import pickle

inference_bundle = {
    "model_state_dict": model.state_dict(),
    "word2idx": word2idx,
    "max_seq_len": MAX_SEQ_LEN,        # 309, per your data
    "vocab_size": VOCAB_SIZE,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
}

torch.save(inference_bundle, "sentiment_rnn_inference.pt")

In [30]:
class SentimentPredictor:
  def __init__(self, bundle_path, model_class, device=None):
    self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    bundle = torch.load(bundle_path, map_location=self.device)
    self.word2idx     = bundle["word2idx"]
    self.max_seq_len  = bundle["max_seq_len"]

    self.model = model_class(
      vocab_size       = bundle["vocab_size"],
      embedding_dim    = bundle["embedding_dim"],
      hidden_dim       = bundle["hidden_dim"],
      embedding_matrix = torch.zeros(bundle["vocab_size"], bundle["embedding_dim"]),  # overwritten below
    )
    self.model.load_state_dict(bundle["model_state_dict"])
    self.model.to(self.device)
    self.model.eval()  # CRITICAL — disables dropout for inference

    self.stop_words = set(stopwords.words("english"))
    self.lemmatizer = WordNetLemmatizer()

  def _clean_text(self, text: str) -> str:
    text = re.sub(r"<.*?>", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

  def _tokenize(self, text: str) -> list[str]:
    tokens = nltk.word_tokenize(self._clean_text(text))
    tokens = [self.lemmatizer.lemmatize(t) for t in tokens if t not in self.stop_words]
    tokens = [t for t in tokens if len(t) > 1]
    return tokens

  def _encode(self, tokens: list[str]) -> list[int]:
    unk_idx = self.word2idx.get("<UNK>", 1)
    return [self.word2idx.get(tok, unk_idx) for tok in tokens]

  def _pad(self, seq: list[int]) -> tuple[list[int], int]:
    true_len = max(len(seq), 1)  # avoid zero-length sequence error in pack_padded_sequence
    if len(seq) > self.max_seq_len:
      seq = seq[:self.max_seq_len]
      true_len = self.max_seq_len
    else:
      seq = seq + [0] * (self.max_seq_len - len(seq))
    return seq, true_len

  @torch.no_grad()
  def predict(self, text: str) -> dict:
    tokens = self._tokenize(text)
    encoded = self._encode(tokens)
    padded, length = self._pad(encoded)

    X = torch.tensor([padded], dtype=torch.long).to(self.device)
    lengths = torch.tensor([length], dtype=torch.long)

    logits = self.model(X, lengths)
    prob = torch.sigmoid(logits).item()
    label = "positive" if prob >= 0.5 else "negative"

    return {
      "text": text,
      "sentiment": label,
      "confidence": prob if label == "positive" else 1 - prob,
      "raw_probability": prob
    }

In [ ]:
predictor = SentimentPredictor("sentiment_rnn_inference.pt", SentimentRNN)

result = predictor.predict("This movie was absolutely fantastic, the acting was superb!")
print(result)
# {'text': '...', 'sentiment': 'positive', 'confidence': 0.91, 'raw_probability': 0.91}

result2 = predictor.predict("Waste of two hours. The plot made no sense and the acting was wooden.")
print(result2)

In [ ]:
tricky_examples = [
    "This movie turned out to be amazing.",   # positive + long-range flip
    "It was very bad, I didn't like it at all.",                                  # double negation
    "The first hour was boring but the ending completely redeemed it.",      # sentiment shift mid-review
]

for text in tricky_examples:
    print(predictor.predict(text))